# X-RecSys on Bluesky Data — INC Ratio Analysis
**CS 294 Final — Bluesky Comparison (Yushan)**

**Experiment:** Feed Bluesky social graph + engagement data into X's open-source
recommendation pipeline (Thunder retrieval → Phoenix retrieval → WeightedScorer),
then measure the in-network content (INC) ratio of the output feed.

**Research question:** Does X's architecture suppress in-network content regardless
of whose data you feed it, or is the 27.5% INC we observed specific to X's user base?

**Dataset:** Failla & Rossetti, Bluesky Social Dataset v3
Zenodo DOI: `10.5281/zenodo.14669616`

**Files used:**
- `followers.csv.gz` (491 MB) — social graph
- `interactions.csv.gz` (1.0 GB) — engagement history + global post corpus proxy
_(We skip user_posts.tar.gz (19.5 GB) — interactions give us a representative post corpus)_

## Section 1 — Config & Constants

In [ ]:
from pathlib import Path
import io, gzip as _gzip, tarfile, urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
from tqdm.auto import tqdm

# ================================================================
# COLUMN MAPPINGS — confirmed from schema probe
# Both files have NO header row; columns are referenced by position.
# ================================================================

# followers.csv.gz  →  col 0 = follower int ID,  col 1 = followee int ID
COL_FOLLOWER = 0
COL_FOLLOWEE = 1

# interactions.csv.gz  →  6 columns, 3 are always null
# col 0 = actor (user who interacted)
# col 1 = null
# col 2 = null
# col 3 = subject author (author of the post that was interacted with)
# col 4 = null
# col 5 = timestamp  (format: YYYYMMDDHHMI  e.g. 202309192352)
COL_ACTOR     = 0
COL_SUBJ_AUTH = 3
COL_IXN_TIME  = 5

# --- Pipeline parameters (mirror X's architecture) ---
THUNDER_K   = 100    # in-network candidates (recency-sorted, bounded by follow list)
PHOENIX_K   = 1500   # out-of-network candidates (global corpus, engagement-sorted)
FINAL_K     = 20     # final ranked feed size (matches X observation study)
HISTORY_LEN = 128    # engagement history window X uses for user token
OON_PENALTY = 0.5    # WeightedScorer OONScorer multiplier for out-of-network posts

# --- Sampling ---
N_SAMPLE_USERS = 500
MATCH_LO       = 67    # ±50% bracket around X participant P1 (133 follows)
MATCH_HI       = 1500  # ±50% bracket around X participant P3 (999 follows)
RANDOM_SEED    = 42

# --- X baseline (from midterm feed observation study) ---
X_INC_MEAN = 0.275
X_PARTICIPANTS = [
    {'id': 'P1', 'follows': 133,  'inc': 0.10},
    {'id': 'P2', 'follows': 644,  'inc': 0.20},
    {'id': 'P3', 'follows': 999,  'inc': 0.35},
    {'id': 'P4', 'follows': 911,  'inc': 0.45},
]

# --- Paths ---
DATA            = Path('data'); DATA.mkdir(exist_ok=True)
FOLLOWERS_GZ    = DATA / 'followers.csv.gz'
INTERACTIONS_GZ = DATA / 'interactions.csv.gz'
ZENODO          = 'https://zenodo.org/records/14669616/files'

print('Config loaded.')

## Section 2 — Download Data

In [ ]:
import os, requests

FILES = {
    FOLLOWERS_GZ:    ("https://zenodo.org/records/14669616/files/followers.csv.gz",    460_000_000),
    INTERACTIONS_GZ: ("https://zenodo.org/records/14669616/files/interactions.csv.gz", 900_000_000),
}

def download_file(url, dest, min_bytes):
    if dest.exists() and dest.stat().st_size >= min_bytes:
        print(f"  ✓ {dest.name}  ({dest.stat().st_size/1e6:.0f} MB, complete)")
        return
    if dest.exists():
        print(f"  ✗ {dest.name} partial ({dest.stat().st_size/1e6:.0f} MB) — re-downloading")
        dest.unlink()
    print(f"  ↓ {dest.name} ...")
    with requests.get(url, stream=True, allow_redirects=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                bar.update(len(chunk))
    size = dest.stat().st_size
    if size < min_bytes:
        raise RuntimeError(f"Incomplete: {size/1e6:.0f} MB < expected {min_bytes/1e6:.0f} MB")
    print(f"  ✓ done  ({size/1e6:.0f} MB)")

for dest, (url, min_bytes) in FILES.items():
    download_file(url, dest, min_bytes)

## Section 2b — Schema Probe
**Run this before anything else.** Print the actual column names from both files.
If they differ from the `COL_*` variables in Section 1, update Section 1 and rerun.

In [ ]:
def peek_gz(path, n=3):
    """Stream first n rows — no full decompression into RAM."""
    df = pd.read_csv(path, compression='gzip', header=None, nrows=n)
    print(f'\n=== {path.name} ===')
    print(df.to_string())
    print('Columns:', list(df.columns))
    return df

peek_gz(FOLLOWERS_GZ)
peek_gz(INTERACTIONS_GZ)

## Section 3 — Build Follow Graph
Stream `followers.csv.gz` in chunks to avoid OOM (491 MB compressed → ~2 GB uncompressed).
Output: `follow_map[user_did]` = frozenset of DIDs they follow.

In [ ]:
print('Building follow graph (chunked) ...')
follow_map_raw: dict[int, set] = defaultdict(set)

for chunk in pd.read_csv(
        FOLLOWERS_GZ,
        compression='gzip',
        header=None,           # no header row
        usecols=[COL_FOLLOWER, COL_FOLLOWEE],
        dtype={COL_FOLLOWER: 'int32', COL_FOLLOWEE: 'int32'},
        chunksize=500_000):
    for row in chunk.itertuples(index=False):
        follow_map_raw[row[0]].add(row[1])

follow_map: dict[int, frozenset] = {
    u: frozenset(s) for u, s in follow_map_raw.items()
}
del follow_map_raw

follow_counts = pd.Series({u: len(s) for u, s in follow_map.items()})
print(f'Follow graph: {len(follow_map):,} users')
print(follow_counts.describe().round(1))

## Section 4 — Load Interactions
Stream `interactions.csv.gz` to build two indexes:
1. `engagement_history[user]` — last 128 interactions per user (for the X user token)
2. `post_corpus[post_uri]` — global post index: author + timestamp + engagement count
   (proxy for the global corpus Phoenix would search)

In [ ]:
from collections import deque

# engagement_history[user] = deque of (author_id, timestamp), max len 128
engagement_history: dict[int, deque] = defaultdict(lambda: deque(maxlen=HISTORY_LEN))
# post_corpus[(author_id, ts)] = engagement count
# Key insight: col3 = author of the post that was interacted with.
# We treat each (actor, author, ts) row as evidence of a "post" by `author` at time `ts`.
post_corpus: dict[tuple, dict] = {}

print('Loading interactions (chunked) ...')
total_rows = 0

for chunk in pd.read_csv(
        INTERACTIONS_GZ,
        compression='gzip',
        header=None,                          # no header row
        usecols=[COL_ACTOR, COL_SUBJ_AUTH, COL_IXN_TIME],
        dtype={COL_ACTOR: 'Int32', COL_SUBJ_AUTH: 'Int32'},  # Int32 tolerates NaN
        chunksize=500_000):

    # Drop rows where actor or subject author is null
    chunk = chunk.dropna(subset=[COL_ACTOR, COL_SUBJ_AUTH])
    total_rows += len(chunk)

    for row in chunk.itertuples(index=False):
        actor  = int(row[0])
        author = int(row[1])
        ts     = row[2]   # raw timestamp string e.g. "202309192352"

        # Engagement history: this actor interacted with content from `author`
        engagement_history[actor].append((author, ts))

        # Global post corpus: track how many times this author's content was engaged with
        key = (author, str(ts)[:12])   # deduplicate by author + hour
        if key not in post_corpus:
            post_corpus[key] = {'author': author, 'ts': ts, 'engagements': 0}
        post_corpus[key]['engagements'] += 1

print(f'Processed {total_rows:,} interaction rows')
print(f'Users with engagement history: {len(engagement_history):,}')
print(f'Unique posts in global corpus: {len(post_corpus):,}')

engagement_history = dict(engagement_history)
all_post_keys = list(post_corpus.keys())
print(f'Global corpus ready ({len(all_post_keys):,} posts)')

## Section 5 — Sample Users
Keep users who have both follow data and engagement history.
Restrict to follow counts in [MATCH_LO, MATCH_HI] to mirror X participants.

In [ ]:
# Users with follow data AND interaction history
eligible = (
    follow_counts[
        follow_counts.between(MATCH_LO, MATCH_HI)
    ]
    .index
    .intersection(engagement_history.keys())
)
print(f'Eligible users (follow count [{MATCH_LO},{MATCH_HI}] + have history): {len(eligible):,}')

rng = np.random.default_rng(RANDOM_SEED)
n = min(N_SAMPLE_USERS, len(eligible))
sampled_users = rng.choice(list(eligible), size=n, replace=False).tolist()
print(f'Sampled {n} users for pipeline.')

# Follow count distribution of sampled users vs X participants
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(follow_counts[sampled_users], bins=30, color='#5C85D6', alpha=0.7,
        label='Bluesky sample')
for p in X_PARTICIPANTS:
    ax.axvline(p['follows'], color='#1DA1F2', linestyle='--', linewidth=1.5,
               label=f"X {p['id']} ({p['follows']})")
ax.set_xlabel('Follow count'); ax.set_ylabel('# users')
ax.set_title('Follow-count distribution: Bluesky sample vs X participants')
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

## Section 6 — Thunder Retrieval (In-Network, Recency-Sorted)
Thunder returns the K most recent posts from accounts the user follows.
No ML, no ranking — pure recency heuristic. INC = 100% at this stage by construction.

In [ ]:
def thunder_retrieve(user: int, k: int = THUNDER_K) -> list[dict]:
    """
    Return up to k most recent posts authored by accounts `user` follows.
    Mirrors Thunder: no ML, recency-sorted, bounded by follow list size.
    INC = 100% here by construction.
    """
    follows = follow_map.get(user, frozenset())
    candidates = [
        {'author': meta['author'], 'ts': meta['ts'],
         'engagements': meta['engagements'], 'in_network': True}
        for meta in post_corpus.values()
        if meta['author'] in follows
    ]
    # Sort by raw timestamp string descending (YYYYMMDDHHMI sorts lexicographically)
    candidates.sort(key=lambda x: str(x['ts'] or ''), reverse=True)
    return candidates[:k]


# Sanity check
u0 = sampled_users[0]
t0 = thunder_retrieve(u0)
print(f'Thunder for user {u0}:')
print(f'  follow count:       {len(follow_map.get(u0, []))}')
print(f'  Thunder candidates: {len(t0)}')
in_net = sum(1 for c in t0 if c['in_network'])
pct = in_net / len(t0) if t0 else 0
print(f'  In-network: {in_net}/{len(t0)} = {pct:.0%}  (should be 100%)')

## Section 7 — Phoenix Retrieval (Global Corpus, Simulated)
Phoenix is a Grok-based two-tower ANN — we cannot run the actual model without weights.
We approximate it by scoring all out-of-network posts using a proxy engagement signal:
`phoenix_score = engagements * recency_decay`
and returning the top PHOENIX_K. This preserves the structural property that matters:
Phoenix searches the **entire global corpus** at full ML capacity,
while Thunder is bounded by follow list size.

In [ ]:
import time

print('Building corpus DataFrame for Phoenix scoring ...')
corpus_df = pd.DataFrame([
    {'key': k, 'author': v['author'], 'ts_raw': v['ts'], 'engagements': v['engagements']}
    for k, v in post_corpus.items()
])

# Parse timestamp: format is YYYYMMDDHHMI (e.g. 202309192352 → 2023-09-19 23:52)
corpus_df['ts'] = pd.to_datetime(
    corpus_df['ts_raw'].astype(str).str[:12],
    format='%Y%m%d%H%M',
    errors='coerce',
    utc=True
)

# Recency score: exponential decay, 7-day half-life
now_ts = pd.Timestamp.utcnow()
days_old = (now_ts - corpus_df['ts']).dt.total_seconds() / 86400
corpus_df['recency']       = np.exp(-0.693 * days_old / 7).fillna(0.0)
corpus_df['phoenix_score'] = corpus_df['engagements'] * corpus_df['recency']

print(f'Corpus: {len(corpus_df):,} posts')
print(corpus_df[['engagements', 'recency', 'phoenix_score']].describe().round(3))


def phoenix_retrieve(user: int, k: int = PHOENIX_K) -> list[dict]:
    """
    Return up to k highest-scoring out-of-network posts.
    Mirrors Phoenix: searches entire global corpus with no follow-list constraint.
    """
    follows = follow_map.get(user, frozenset())
    oon_mask = ~corpus_df['author'].isin(follows)
    top = corpus_df[oon_mask].nlargest(k, 'phoenix_score')
    return [
        {'author': int(row.author), 'ts': row.ts,
         'engagements': int(row.engagements), 'in_network': False}
        for row in top.itertuples()
    ]


# Sanity check
u0 = sampled_users[0]
p0 = phoenix_retrieve(u0)
print(f'\nPhoenix for user {u0}: {len(p0)} candidates, '
      f'{sum(1 for c in p0 if c["in_network"])}/{len(p0)} in-network (should be 0)')

## Section 8 — WeightedScorer Proxy Ranking
X's WeightedScorer: `Score = Σ(wᵢ · P(actionᵢ)) + offset`
Out-of-network posts receive an OONScorer penalty multiplier.

Proxy: `score = engagements_normalized + recency_bonus`,
then `score *= OON_PENALTY` for out-of-network candidates.
Combine Thunder + Phoenix candidates, rank, take top FINAL_K.

In [ ]:
def weighted_score(candidate: dict, max_eng: float) -> float:
    """Proxy for X's WeightedScorer engagement prediction."""
    eng_norm   = candidate['engagements'] / (max_eng + 1)
    # Recency bonus: same decay as Phoenix scoring
    if candidate['ts'] is not None and not pd.isnull(candidate['ts']):
        ts = candidate['ts']
        if hasattr(ts, 'timestamp'):
            age_days = (time.time() - ts.timestamp()) / 86400
        else:
            age_days = 1.0
        recency = np.exp(-0.693 * age_days / 7)
    else:
        recency = 0.5
    raw = eng_norm + 0.3 * recency
    # Apply OONScorer penalty for out-of-network posts
    return raw * (1.0 if candidate['in_network'] else OON_PENALTY)


def simulate_for_you_feed(user: str) -> list[dict]:
    """Run the full X pipeline on one Bluesky user."""
    thunder = thunder_retrieve(user, THUNDER_K)
    phoenix = phoenix_retrieve(user, PHOENIX_K)
    candidates = thunder + phoenix
    if not candidates:
        return []
    max_eng = max(c['engagements'] for c in candidates) or 1
    for c in candidates:
        c['score'] = weighted_score(c, max_eng)
    candidates.sort(key=lambda x: x['score'], reverse=True)
    return candidates[:FINAL_K]


# Sanity check on one user
feed0 = simulate_for_you_feed(u0)
inc0 = sum(1 for p in feed0 if p['in_network']) / len(feed0) if feed0 else 0
print(f'Simulated feed for {u0[:20]}...')
print(f'  Total posts: {len(feed0)}')
print(f'  In-network:  {sum(1 for p in feed0 if p["in_network"])}/{len(feed0)} = {inc0:.0%}')
print(f'  (X group mean for comparison: {X_INC_MEAN:.0%})')

## Section 9 — Run Pipeline on All Sampled Users

In [ ]:
results = []

for user in tqdm(sampled_users, desc='Simulating For You feeds'):
    feed = simulate_for_you_feed(user)
    if not feed:
        continue
    in_net = sum(1 for p in feed if p['in_network'])
    results.append({
        'user': user,
        'follow_count': len(follow_map.get(user, [])),
        'thunder_in_feed': sum(1 for p in feed if p['in_network']),
        'phoenix_in_feed': sum(1 for p in feed if not p['in_network']),
        'inc_ratio': in_net / len(feed),
    })

results_df = pd.DataFrame(results)
print(results_df[['follow_count','inc_ratio']].describe().round(3))

## Section 10 — Results & Figures

In [ ]:
# --- Figure A: INC ratio distribution (Bluesky-through-X-pipeline vs X observed) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: histogram of per-user INC ratios
ax = axes[0]
ax.hist(results_df['inc_ratio'], bins=25, color='#5C85D6', alpha=0.75,
        edgecolor='white', label='Bluesky users through X pipeline')
ax.axvline(results_df['inc_ratio'].mean(), color='#3A5BA0', linewidth=2,
           label=f"Bluesky mean = {results_df['inc_ratio'].mean():.1%}")
ax.axvline(X_INC_MEAN, color='#1DA1F2', linewidth=2, linestyle='--',
           label=f'X group mean = {X_INC_MEAN:.1%}')
ax.set_xlabel('INC ratio (in-network posts / 20)')
ax.set_ylabel('# users')
ax.set_title('INC ratio distribution:\nBluesky data through X algorithm vs X observed')
ax.legend(fontsize=8)

# Panel 2: INC vs follow count — Bluesky + X participants overlaid
ax = axes[1]
ax.scatter(results_df['follow_count'], results_df['inc_ratio'],
           alpha=0.25, s=10, color='#5C85D6', label='Bluesky user (X pipeline)')
# Trend line
z = np.polyfit(results_df['follow_count'], results_df['inc_ratio'], 1)
xs = np.linspace(results_df['follow_count'].min(), results_df['follow_count'].max(), 200)
ax.plot(xs, np.polyval(z, xs), color='#3A5BA0', linewidth=2, label='Bluesky trend')
# X participants
for p in X_PARTICIPANTS:
    ax.scatter(p['follows'], p['inc'], s=150, zorder=6, marker='D',
               color='#1DA1F2', edgecolors='white', linewidth=0.8)
    ax.annotate(p['id'], (p['follows'], p['inc']),
                textcoords='offset points', xytext=(6, 4), fontsize=8, color='#1DA1F2')
ax.axhline(X_INC_MEAN, color='#1DA1F2', linewidth=1.5, linestyle='--',
           label=f'X mean ({X_INC_MEAN:.1%})')
ax.set_xlabel('Follow count'); ax.set_ylabel('INC ratio')
ax.set_title('INC ratio vs follow count:\nBluesky through X pipeline vs X participants')
ax.legend(fontsize=8); ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('bluesky_x_pipeline_inc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Figure B: Candidate pool composition (Thunder vs Phoenix in final feed) ---
thunder_share = results_df['thunder_in_feed'].mean() / FINAL_K
phoenix_share = results_df['phoenix_in_feed'].mean() / FINAL_K

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.barh(['Final feed\ncomposition'],
               [thunder_share, phoenix_share],
               color=['#4CAF50', '#F44336'],
               left=[0, thunder_share],
               height=0.4)
ax.axvline(X_INC_MEAN, color='#1DA1F2', linewidth=2, linestyle='--',
           label=f'X observed INC ({X_INC_MEAN:.1%})')
ax.set_xlim(0, 1); ax.set_xlabel('Fraction of final 20-post feed')
ax.set_title('Thunder (in-network) vs Phoenix (out-of-network) share in final feed')
thunder_patch = mpatches.Patch(color='#4CAF50', label=f'Thunder / in-network ({thunder_share:.1%})')
phoenix_patch = mpatches.Patch(color='#F44336', label=f'Phoenix / out-of-network ({phoenix_share:.1%})')
ax.legend(handles=[thunder_patch, phoenix_patch, bars[0]], fontsize=8)
plt.tight_layout()
plt.savefig('candidate_pool_composition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Print-ready numbers for paper ---
mean_inc  = results_df['inc_ratio'].mean()
med_inc   = results_df['inc_ratio'].median()
std_inc   = results_df['inc_ratio'].std()
delta     = mean_inc - X_INC_MEAN

print('=' * 55)
print('PAPER NUMBERS — X Pipeline on Bluesky Data')
print('=' * 55)
print(f'N users simulated:            {len(results_df)}')
print(f'Follow count range:           {MATCH_LO}–{MATCH_HI}')
print(f'Pipeline: Thunder K={THUNDER_K}, Phoenix K={PHOENIX_K}, Final K={FINAL_K}')
print(f'OON penalty:                  {OON_PENALTY}')
print()
print(f'Bluesky-through-X mean INC:   {mean_inc:.1%}  (σ={std_inc:.1%})')
print(f'Bluesky-through-X median INC: {med_inc:.1%}')
print(f'X observed group mean INC:    {X_INC_MEAN:.1%}')
print(f'Difference (Bluesky – X):     {delta:+.1%}')
print()
print('INC by follow-count quartile:')
results_df['fc_quartile'] = pd.qcut(results_df['follow_count'], 4,
                                     labels=['Q1 (low)','Q2','Q3','Q4 (high)'])
print(results_df.groupby('fc_quartile')['inc_ratio']
      .mean().map('{:.1%}'.format).to_string())
print()
above_x = (results_df['inc_ratio'] > X_INC_MEAN).mean()
print(f'{above_x:.0%} of Bluesky users get INC > X baseline when run through X pipeline')

## Section 11 — Sensitivity Analysis
How much does the OON_PENALTY value change the conclusion?
Sweep OON_PENALTY from 0.1 (heavy suppression) to 1.0 (no penalty).

In [ ]:
# Sweep OON_PENALTY — reuse already-loaded data, just re-rank
penalties = [0.1, 0.25, 0.5, 0.75, 1.0]
sweep_results = {}

for pen in penalties:
    inc_list = []
    for user in sampled_users:
        thunder = thunder_retrieve(user, THUNDER_K)
        phoenix = phoenix_retrieve(user, PHOENIX_K)
        candidates = thunder + phoenix
        if not candidates: continue
        max_eng = max(c['engagements'] for c in candidates) or 1
        for c in candidates:
            eng_norm = c['engagements'] / (max_eng + 1)
            raw = eng_norm  # simplified score for sweep
            c['score'] = raw * (1.0 if c['in_network'] else pen)
        candidates.sort(key=lambda x: x['score'], reverse=True)
        feed = candidates[:FINAL_K]
        inc_list.append(sum(1 for p in feed if p['in_network']) / len(feed))
    sweep_results[pen] = np.mean(inc_list)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(penalties, [sweep_results[p] for p in penalties],
        marker='o', color='#5C85D6', linewidth=2)
ax.axhline(X_INC_MEAN, color='#1DA1F2', linestyle='--', linewidth=1.5,
           label=f'X observed ({X_INC_MEAN:.1%})')
ax.set_xlabel('OON penalty (1.0 = no penalty)')
ax.set_ylabel('Mean INC ratio')
ax.set_title('Sensitivity of INC ratio to OONScorer penalty\n(Bluesky data through X pipeline)')
ax.legend(); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('oon_penalty_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

print('OON penalty → mean INC:')
for pen in penalties:
    flag = ' ← default' if pen == OON_PENALTY else ''
    print(f'  {pen:.2f}  →  {sweep_results[pen]:.1%}{flag}')

## Section 12 — Interpretation Template
Fill in `[values]` from Section 10 output once the analysis runs.

---

### 4.2 Bluesky Comparison

To test whether in-network suppression is specific to X's architecture or an emergent property
of engagement optimization at scale, we ran the same pipeline on a different population:
Bluesky users from the Failla and Rossetti dataset [Zenodo v3, DOI: 10.5281/zenodo.14669616].
Rather than computing INC from Bluesky's own feed generators, we fed Bluesky social graph
and engagement data directly into X's open-source recommendation pipeline — Thunder retrieval,
Phoenix retrieval, and WeightedScorer ranking — and measured what INC ratio the algorithm produces.

**Setup.** We sampled [N] Bluesky users with follow counts between [MATCH_LO] and [MATCH_HI],
bracketing our four X participants (133–999 follows). For each user, Thunder retrieved the
[THUNDER_K] most recent posts from followed accounts; a simulated Phoenix retrieved the
[PHOENIX_K] highest-engagement out-of-network posts from the global corpus (proxy:
engagement count × recency decay). WeightedScorer ranked the combined pool with an OON penalty
of [OON_PENALTY] applied to out-of-network candidates, and the top 20 posts formed the
simulated For You feed.

**Result.** Running X's pipeline on Bluesky data produces a mean INC ratio of **[mean_inc]%**
(σ = [std]%), compared to **27.5%** observed on actual X users (p[X participants]). The
[delta: positive → 'higher' / negative → 'lower'] result held across the follow-count range,
and the INC vs follow-count relationship remained monotonic (Q1: [Q1]%, Q4: [Q4]%),
replicating the Thunder bounded-pool effect seen on X.

**Interpretation.** [Choose based on delta direction:]

*If mean_inc ≈ X_INC_MEAN (within ~5pp):* X's architecture produces roughly the same INC
suppression on Bluesky users as on X's own users. In-network suppression is not a property
of X's specific user base but of the Thunder/Phoenix retrieval asymmetry itself. Any platform
that adopts this architecture would produce similar suppression regardless of its users'
social graph structure.

*If mean_inc > X_INC_MEAN:* Bluesky users receive more in-network content even through X's
pipeline — suggesting their social graphs are more tightly clustered than X's, and Thunder's
bounded pool covers a larger fraction of relevant content. X's 27.5% INC partly reflects
X's specific user-graph topology, not only the algorithm.

*If mean_inc < X_INC_MEAN:* The architecture suppresses in-network content more aggressively
on Bluesky users — their sparser or more diverse follow graphs give Thunder a smaller pool,
amplifying Phoenix's dominance. This strengthens the structural argument: the algorithm
suppresses in-network content by design, and the effect is worse on platforms with
looser social graphs.

The sensitivity sweep (Section 11) shows that [N] of the 5 tested penalty values produce
INC ratios below the X baseline, confirming the result is not an artifact of the OON_PENALTY
choice.